## Temporal Analysis of butterfly species using cosine similarity
### How does mean cosine distance in a species change over time?

#### Imports and setup

In [1]:
import duckdb
import lancedb
import logging
import numpy as np
import os
from dotenv import load_dotenv
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)

NOTEBOOK_ROOT = Path.cwd()
REPO_ROOT = NOTEBOOK_ROOT if (NOTEBOOK_ROOT / "backend").exists() else NOTEBOOK_ROOT.parent
ENV_PATH = REPO_ROOT / "backend" / ".env"
if not ENV_PATH.exists():
    raise FileNotFoundError(f"Backend environment file not found: {ENV_PATH}")
load_dotenv(ENV_PATH)


def env_path(name):
    value = os.getenv(name)
    if not value:
        raise ValueError(f"Missing {name} in {ENV_PATH}")
    path = Path(value).expanduser()
    return path if path.is_absolute() else (ENV_PATH.parent / path).resolve()


DUCK_DIR = env_path("DUCK_DIR")
LANCE_DIR = env_path("LANCE_DIR")
IMAGE_DIR = env_path("IMAGE_DIR")
GBIF_DIR = env_path("GBIF_DIR")
IMAGE_META_DIR = env_path("IMAGE_META_DIR")

DUCK_PATH = DUCK_DIR / "biocosmos.duckdb"
LANCE_PATH = LANCE_DIR / "biocosmos.lance"
LANCE_TABLE = "nymphalidae"
META_TABLE = "image_meta"
MIN_SPECIES_IMAGES = 100
TOP_K = 800
LIMIT = 10
SIDES = ["dorsal", "ventral"]

logger.info("Python interpreter: %s", os.sys.executable)
logger.info("Loaded paths from %s", ENV_PATH)

2026-09-11 16:39:49,030 [INFO] Python interpreter: /Users/kiraluna/Projects/bioVisionLab-Project/v1/BioCosmos/.venv/bin/python
2026-09-11 16:39:49,030 [INFO] Loaded paths from /Users/kiraluna/Projects/bioVisionLab-Project/v1/BioCosmos/backend/.env


#### Validate paths

In [2]:
if not LANCE_PATH.exists():
    raise FileNotFoundError(f"LanceDB database not found: {LANCE_PATH}")
if not DUCK_PATH.exists():
    raise FileNotFoundError(f"DuckDB file not found: {DUCK_PATH}")

logger.info("Using DuckDB file: %s", DUCK_PATH)
logger.info("Using LanceDB database: %s", LANCE_PATH)
logger.info("Using image directory: %s%s", IMAGE_DIR, " (available)" if IMAGE_DIR.exists() else " (not found)")
logger.info("Using GBIF directory: %s%s", GBIF_DIR, " (available)" if GBIF_DIR.exists() else " (not found)")
logger.info("Using image metadata directory: %s%s", IMAGE_META_DIR, " (available)" if IMAGE_META_DIR.exists() else " (not found)")

2026-09-11 16:39:54,441 [INFO] Using DuckDB file: /Users/kiraluna/Projects/bioVisionLab-Project/biocosmos/duck_db/biocosmos.duckdb
2026-09-11 16:39:54,441 [INFO] Using LanceDB database: /Users/kiraluna/Projects/bioVisionLab-Project/biocosmos/lance_db_lite/biocosmos.lance
2026-09-11 16:39:54,442 [INFO] Using image directory: /Users/kiraluna/Projects/bioVisionLab-Project/python/biocosmos-exploration/data/nymphalidae_new (not found)
2026-09-11 16:39:54,442 [INFO] Using GBIF directory: /Users/kiraluna/Projects/bioVisionLab-Project/biocosmos/lep-meta (not found)
2026-09-11 16:39:54,442 [INFO] Using image metadata directory: /Users/kiraluna/Projects/bioVisionLab-Project/biocosmos/new_data (available)


#### Load data into memory

In [3]:
logger.info("Connecting to LanceDB: %s", LANCE_PATH)
lance_db = lancedb.connect(str(LANCE_PATH))
table_names = lance_db.table_names()
logger.info("Available LanceDB tables: %s", table_names)
if LANCE_TABLE not in table_names:
    raise ValueError(f"LanceDB table '{LANCE_TABLE}' not found; available tables: {table_names}")
lance_table = lance_db.open_table(LANCE_TABLE)
logger.info("LanceDB table '%s': %s rows", LANCE_TABLE, lance_table.count_rows())

duck_connection = duckdb.connect(str(DUCK_PATH), read_only=True)
logger.info("Connected to DuckDB: %s", DUCK_PATH)

2026-09-11 16:39:54,497 [INFO] Connecting to LanceDB: /Users/kiraluna/Projects/bioVisionLab-Project/biocosmos/lance_db_lite/biocosmos.lance
2026-09-11 16:39:54,511 [INFO] Available LanceDB tables: ['nymphalidae']
2026-09-11 16:39:54,541 [INFO] LanceDB table 'nymphalidae': 619787 rows
2026-09-11 16:39:54,549 [INFO] Connected to DuckDB: /Users/kiraluna/Projects/bioVisionLab-Project/biocosmos/duck_db/biocosmos.duckdb


#### Load image metadata and embeddings

In [6]:
def normalize_species_name(species):
    """Normalize species names for matching database values."""
    if species is None:
        return None
    return species.strip().lower().replace(" ", "_")


def load_metadata(duck_connection, species=None):
    """Load metadata, excluding species with fewer than MIN_SPECIES_IMAGES images."""
    normalized_species = normalize_species_name(species)
    logger.info("Loading image metadata from DuckDB%s...",
                f" for '{normalized_species}'" if normalized_species else "")
    normalized_species_sql = "REPLACE(LOWER(species), ' ', '_')"
    query = f"""
        WITH metadata AS (
            SELECT img_id,
                   {normalized_species_sql} AS species,
                   LOWER(class_dv) AS side,
                   COUNT(*) OVER (
                       PARTITION BY {normalized_species_sql}
                   ) AS species_image_count
            FROM {META_TABLE}
        )
        SELECT img_id, species, side
        FROM metadata
        WHERE species_image_count >= ?
    """
    parameters = [MIN_SPECIES_IMAGES]
    if normalized_species:
        query += " AND species = ?"
        parameters.append(normalized_species)
    meta = duck_connection.execute(query, parameters).pl()
    if normalized_species and meta.is_empty():
        raise ValueError(
            f"Species '{species}' was not found or has fewer than "
            f"{MIN_SPECIES_IMAGES} images"
        )
    logger.info("Loaded %d metadata rows", len(meta))
    return meta


def load_all_embeddings(lance_table, img_ids=None):
    """Load and L2-normalize LanceDB embeddings for selected image IDs."""
    logger.info("Loading embeddings from LanceDB%s...",
                f" for {len(img_ids)} metadata image IDs" if img_ids is not None else "")
    arrow_table = lance_table.to_arrow().select(["img_id", "unicom_embeddings"])
    if img_ids is not None:
        wanted_ids = set(np.asarray(img_ids).tolist())
        stored_ids = np.asarray(arrow_table["img_id"])
        mask = np.array([img_id in wanted_ids for img_id in stored_ids], dtype=bool)
        arrow_table = arrow_table.filter(mask)
    loaded_img_ids = np.asarray(arrow_table["img_id"])
    embeddings = np.asarray(
        arrow_table["unicom_embeddings"].to_pylist(),
        dtype=np.float32,
    )
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    embeddings /= norms
    if img_ids is not None and len(loaded_img_ids) == 0:
        raise ValueError("No metadata image IDs were found in the LanceDB table")
    logger.info("Loaded %d embeddings with shape %s", len(loaded_img_ids), embeddings.shape)
    return embeddings, loaded_img_ids


# Set to None to load all qualifying species, or use a normalized name such as
# "lysandra_coridon" to load only one qualifying species.
SPECIES_FILTER = "lysandra_coridon"
meta = load_metadata(duck_connection, SPECIES_FILTER)
embeddings, img_ids = load_all_embeddings(lance_table, meta["img_id"].to_numpy())

2026-09-11 16:42:25,361 [INFO] Loading image metadata from DuckDB for 'lysandra_coridon'...
2026-09-11 16:42:25,466 [INFO] Loaded 21739 metadata rows
2026-09-11 16:42:25,469 [INFO] Loading embeddings from LanceDB for 21739 metadata image IDs...
2026-09-11 16:42:30,071 [INFO] Loaded 21739 embeddings with shape (21739, 768)


#### Next step
* Take centroid (average) and compute distances to each image from there (precompute_similarity.py)
* Data analysis over time
    * Use scikit-learn?
    * Graph average centroid distance (distance, y axis) per month (time, x axis) 
* Test on individual species to preview and modify graph(s) and visualizations